# **Dataset Loader**

In [ ]:
import os
import cv2
import numpy as np

IMG_SIZE = 224

def load_dataset(base_path):
    X, y = [], []

    for label, folder in enumerate(["intact", "damaged"]):
        folder_path = os.path.join(base_path, folder)

        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0

            X.append(img)
            y.append(label)

    return np.array(X), np.array(y)

# **CNN Feature Extractor (ResNet50)**

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input

def build_cnn_feature_extractor():
    base = ResNet50(weights='imagenet', include_top=False,
                    input_tensor=Input(shape=(224,224,3)))

    x = GlobalAveragePooling2D()(base.output)
    model = Model(inputs=base.input, outputs=x)

    return model

# **Feature Extraction Script**

In [ ]:
base_path = r"D:\parcel_damage_classification\media\damaged-and-intact-packages"

X, y = load_dataset(base_path)

cnn_model = build_cnn_feature_extractor()
features = cnn_model.predict(X, batch_size=32)

np.save("features.npy", features)
np.save("labels.npy", y)

print("Feature extraction completed")


# **Train SVM Classifier**

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import numpy as np

X = np.load("features.npy")
y = np.load("labels.npy")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

joblib.dump(svm, "svm_model.pkl")

# **Prediction Script**

In [ ]:
import cv2
import numpy as np
import joblib
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Input

IMG_SIZE = 224

svm = joblib.load("svm_model.pkl")

base = ResNet50(weights='imagenet', include_top=False,
                input_tensor=Input(shape=(224,224,3)))
cnn_model = Model(inputs=base.input,
                  outputs=GlobalAveragePooling2D()(base.output))


def predict_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    feature = cnn_model.predict(img)
    pred = svm.predict(feature)[0]
    prob = svm.predict_proba(feature)[0]

    label = "DAMAGED" if pred == 1 else "INTACT"

    print(f"Prediction: {label}")
    print(f"Confidence: {np.max(prob)*100:.2f}%")


predict_image("test.jpg")